In [ ]:
import json
import os
import time
import urllib.request
import pandas as pd

# Output folders
CSV_DIR  = "gen8ou-1825"
JSON_DIR = "oujsons"
os.makedirs(CSV_DIR,  exist_ok=True)
os.makedirs(JSON_DIR, exist_ok=True)

# All months to scrape
MONTHS = [
    "2021-01", "2021-02", "2021-03", "2021-04", "2021-05", "2021-06",
    "2021-07", "2021-08", "2021-09", "2021-10", "2021-11", "2021-12",
    "2022-01", "2022-02", "2022-03", "2022-04", "2022-05", "2022-06",
    "2022-07", "2022-08", "2022-09", "2022-10",
]

FORMAT = "gen8ou"
RATING  = "1825"
BASE_URL = "https://www.smogon.com/stats/{month}/chaos/{format}-{rating}.json"

OUTPUT_CSV = os.path.join(CSV_DIR, "gen8ou_1825_usage_timeseries.csv")

rows = []

for month in MONTHS:
    url = BASE_URL.format(month=month, format=FORMAT, rating=RATING)
    print(f"Fetching {url} ...", end=" ")

    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=30) as resp:
            raw = resp.read().decode("utf-8")
            obj = json.loads(raw)
    except Exception as e:
        print(f"FAILED ({e})")
        continue

    # Save raw JSON
    json_path = os.path.join(JSON_DIR, f"gen8ou-1825-{month}.json")
    with open(json_path, "w", encoding="utf-8") as jf:
        jf.write(raw)
''
    data           = obj.get("data", {})
    total_battles  = obj.get("info", {}).get("number of battles", 0) or 0

    for mon, info in data.items():
        if not isinstance(info, dict):
            continue

        usage     = float(info.get("usage",     0) or 0)
        raw_count = float(info.get("Raw count", 0) or 0)
        real_pct  = (raw_count / total_battles) if total_battles else 0.0

        rows.append({
            "month":         month,
            "pokemon":       mon,
            "usage":         usage,          # Glicko2-weighted usage %
            "raw_count":     raw_count,      # raw appearances
            "real_usage":    real_pct,       # raw_count / total_battles
            "total_battles": total_battles,
        })

    print(f"OK ({len(data)} pokemon, {total_battles} battles)")
    time.sleep(0.5)  # be polite

df = pd.DataFrame(rows, columns=[
    "month", "pokemon", "usage", "raw_count", "real_usage", "total_battles"
])

df.sort_values(["pokemon", "month"], inplace=True)
df.to_csv(OUTPUT_CSV, index=False)

print(f"\n✅ Saved {len(df)} rows → {OUTPUT_CSV}")
print(f"   Months scraped : {df['month'].nunique()}")
print(f"   Unique pokemon : {df['pokemon'].nunique()}")

Fetching https://www.smogon.com/stats/2021-01/chaos/gen8ou-1825.json ... OK (272 pokemon, 1804169 battles)
Fetching https://www.smogon.com/stats/2021-02/chaos/gen8ou-1825.json ... OK (264 pokemon, 1485104 battles)
Fetching https://www.smogon.com/stats/2021-03/chaos/gen8ou-1825.json ... OK (262 pokemon, 1628081 battles)
Fetching https://www.smogon.com/stats/2021-04/chaos/gen8ou-1825.json ... OK (240 pokemon, 1398223 battles)
Fetching https://www.smogon.com/stats/2021-05/chaos/gen8ou-1825.json ... OK (259 pokemon, 1345731 battles)
Fetching https://www.smogon.com/stats/2021-06/chaos/gen8ou-1825.json ... OK (274 pokemon, 1336739 battles)
Fetching https://www.smogon.com/stats/2021-07/chaos/gen8ou-1825.json ... OK (237 pokemon, 1558433 battles)
Fetching https://www.smogon.com/stats/2021-08/chaos/gen8ou-1825.json ... OK (288 pokemon, 1506862 battles)
Fetching https://www.smogon.com/stats/2021-09/chaos/gen8ou-1825.json ... OK (284 pokemon, 1467367 battles)
Fetching https://www.smogon.com/stats